In [3]:
import ollama

print(ollama.list())

{'models': [{'name': 'llama3:latest', 'model': 'llama3:latest', 'modified_at': '2025-09-17T13:24:30.560532183-04:00', 'size': 4661224676, 'digest': '365c0bd3c000a25d28ddbf732fe1c6add414de7275464c4e4d1c3b5fcb5d8ad1', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '8.0B', 'quantization_level': 'Q4_0'}}, {'name': 'mistral:latest', 'model': 'mistral:latest', 'modified_at': '2025-07-16T12:46:24.54017068-04:00', 'size': 4372824384, 'digest': '6577803aa9a036369e481d648a2baebb381ebc6e897f2bb9a766a2aa7bfbc1cf', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '7.2B', 'quantization_level': 'Q4_K_M'}}, {'name': 'qwen2.5:latest', 'model': 'qwen2.5:latest', 'modified_at': '2025-04-02T16:06:38.932341123-04:00', 'size': 4683087332, 'digest': '845dbda0ea48ed749caafd9e6037047aa19acfcfd82e704d7ca97d631a0b697e', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen2', 'f

In [2]:
import json
import re
import time
import csv
from pathlib import Path
import pandas as pd


In [25]:
SYSTEM_INSTRUCTIONS = (
    "You are an information extraction engine. "
    "You will be given a paragraph describing hourly weather observations. "
    "Extract every hourly record as JSON array. "
    "For each record, return these fields with numbers only (floats) where applicable:\n"
    "- datetime_iso (ISO 8601),\n"
    "- city,\n"
    "- province,\n"
    "- u10_ms (east-west wind at 10 m, m/s),\n"
    "- v10_ms (north-south wind at 10 m, m/s),\n"
    "- dewpoint_c,\n"
    "- temperature_c,\n"
    "- p_msl_hpa,\n"
    "- p_surf_hpa,\n"
    "- precip_m.\n"
    "Output strictly as JSON with the shape: {\"records\": [ ... ]}. "
    "Do not include any commentary, markdown, or code fences."
)

In [4]:
def build_prompt(text: str) -> str:
    return (
        f"{SYSTEM_INSTRUCTIONS}\n\n"
        "Text:\n"
        f"\"\"\"\n{text}\n\"\"\"\n\n"
        "Return JSON now."
    )

In [5]:
def call_ollama(prompt: str, MODEL_NAME, temperature: float = 0.0, retries: int = 3):

    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTIONS},
        {"role": "user", "content": prompt},
    ]
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            resp = ollama.chat(
                model=MODEL_NAME,
                messages=messages,
                options={"temperature": temperature},
                stream=False,
            )
            # format response ollama.chat: {'message': {'role': 'assistant', 'content': '...'}, ...}
            return resp["message"]["content"]
        except Exception as e:
            last_err = e
            # backoff sederhana
            time.sleep(1.5 * attempt)
    raise RuntimeError(f"Ollama chat failed after {retries} attempts: {last_err}")

In [6]:
def coerce_json(text: str):
    """
    Usaha mem-parse balasan menjadi JSON dict.
    Menghapus kemungkinan code fence & mengambil objek { ... } terluar.
    """
    cleaned = re.sub(r"^```(json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start != -1 and end != -1 and end > start:
        cleaned = cleaned[start : end + 1]
    return json.loads(cleaned)

In [7]:
def extract_from_response_text(text: str, model:str):
    """Bangun prompt -> panggil LLM -> parse JSON -> normalisasi."""
    prompt = build_prompt(text)
    raw = call_ollama(prompt, model)
    parsed = coerce_json(raw)
    if not isinstance(parsed, dict) or "records" not in parsed or not isinstance(parsed["records"], list):
        raise ValueError("Unexpected JSON shape from model.")
    norm = []
    for rec in parsed["records"]:
        def fget(k):
            v = rec.get(k, None)
            if v is None or v == "":
                return None
            try:
                return float(v)
            except Exception:
                return v
        norm.append({
            "datetime_iso": rec.get("datetime_iso"),
            "city": rec.get("city"),
            "province": rec.get("province"),
            "u10_ms": fget("u10_ms"),
            "v10_ms": fget("v10_ms"),
            "dewpoint_c": fget("dewpoint_c"),
            "temperature_c": fget("temperature_c"),
            "p_msl_hpa": fget("p_msl_hpa"),
            "p_surf_hpa": fget("p_surf_hpa"),
            "precip_m": fget("precip_m"),
        })
    return norm

In [8]:
def read_jsonl(path: str):
    """Baca file .jsonl yang tiap baris berisi objek JSON dengan kolom 'response'."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            yield obj.get("LLM Result", "")

In [9]:
def write_csv(rows, out_path: str):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        "datetime_iso",
        "city",
        "province",
        "u10_ms",
        "v10_ms",
        "dewpoint_c",
        "temperature_c",
        "p_msl_hpa",
        "p_surf_hpa",
        "precip_m",
    ]
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow(r)

In [10]:
def main(input_jsonl: str, output_csv: str, model: str):
    all_rows = []
    for i, response_text in enumerate(read_jsonl(input_jsonl), start=1):
        if not response_text:
            continue
        try:
            records = extract_from_response_text(response_text, model)
            all_rows.extend(records)
            print(f"[OK] line {i}: extracted {len(records)} record(s)")
        except Exception as e:
            print(f"[WARN] line {i}: failed to extract -> {e}")
    if not all_rows:
        print("No records extracted.")
        return
    write_csv(all_rows, output_csv)
    print(f"Saved {len(all_rows)} records to {output_csv}")


In [18]:
#converting csv to jsonl

import os
import csv
import json

def csv_to_jsonl(src_folder, dst_folder, filenames=None):

    os.makedirs(dst_folder, exist_ok=True)

    if filenames is None:
        filenames = [f for f in os.listdir(src_folder) if f.endswith(".csv")]

    for fname in filenames:
        src_path = os.path.join(src_folder, fname)
        dst_path = os.path.join(dst_folder, os.path.splitext(fname)[0] + ".jsonl")

        with open(src_path, newline='', encoding='utf-8') as csvfile, \
             open(dst_path, 'w', encoding='utf-8') as jsonlfile:

            reader = csv.DictReader(csvfile)
            for row in reader:
                jsonlfile.write(json.dumps(row, ensure_ascii=False) + "\n")

        print(f"Converted: {src_path} → {dst_path}")

# Example usage:
src_folder = "data/5_Clusters_1_SubCluster_PCA/Result"
dst_folder = "data/5_Clusters_1_SubCluster_PCA/Result"

# Convert all CSVs in the folder
csv_to_jsonl(src_folder, dst_folder)

# Or convert only selected ones
# csv_to_jsonl(src_folder, dst_folder, filenames=["file1.csv", "file2.csv"])


Converted: data/5_Clusters_1_SubCluster_PCA/Result/weather_llama2_experiment_log_20250708_113251.csv → data/5_Clusters_1_SubCluster_PCA/Result/weather_llama2_experiment_log_20250708_113251.jsonl
Converted: data/5_Clusters_1_SubCluster_PCA/Result/weather_mistral_experiment_log_20250708_162525.csv → data/5_Clusters_1_SubCluster_PCA/Result/weather_mistral_experiment_log_20250708_162525.jsonl
Converted: data/5_Clusters_1_SubCluster_PCA/Result/weather_qwen2.5_experiment_log_20250708_165623.csv → data/5_Clusters_1_SubCluster_PCA/Result/weather_qwen2.5_experiment_log_20250708_165623.jsonl
Converted: data/5_Clusters_1_SubCluster_PCA/Result/weather_mistral_experiment_log_20250715_070455.csv → data/5_Clusters_1_SubCluster_PCA/Result/weather_mistral_experiment_log_20250715_070455.jsonl
Converted: data/5_Clusters_1_SubCluster_PCA/Result/weather_qwen2.5_experiment_log_20250708_143602.csv → data/5_Clusters_1_SubCluster_PCA/Result/weather_qwen2.5_experiment_log_20250708_143602.jsonl
Converted: data/5

In [38]:
if __name__ == "__main__":
    main(input_jsonl="data/5_Clusters_1_SubCluster_PCA/Result/weather_llama2_experiment_log_20250708_113251.jsonl", output_csv="data/5_Clusters_1_SubCluster_PCA/Eval", model="mistral")

[OK] line 1: extracted 3 record(s)
[OK] line 2: extracted 1 record(s)
[OK] line 3: extracted 6 record(s)
[OK] line 4: extracted 5 record(s)
[OK] line 5: extracted 5 record(s)
[OK] line 6: extracted 3 record(s)
[OK] line 7: extracted 4 record(s)
[OK] line 8: extracted 4 record(s)
[OK] line 9: extracted 4 record(s)
[OK] line 10: extracted 1 record(s)
[OK] line 11: extracted 5 record(s)
[OK] line 12: extracted 1 record(s)
[OK] line 13: extracted 1 record(s)
[OK] line 14: extracted 5 record(s)
[OK] line 15: extracted 5 record(s)
[OK] line 16: extracted 2 record(s)
[OK] line 17: extracted 5 record(s)
[OK] line 18: extracted 4 record(s)
[OK] line 19: extracted 2 record(s)
[OK] line 20: extracted 6 record(s)
[OK] line 21: extracted 5 record(s)
[OK] line 22: extracted 4 record(s)
[OK] line 23: extracted 1 record(s)
[OK] line 24: extracted 5 record(s)
[OK] line 25: extracted 1 record(s)
[OK] line 26: extracted 2 record(s)
[OK] line 27: extracted 6 record(s)
[OK] line 28: extracted 5 record(s)
[

IsADirectoryError: [Errno 21] Is a directory: 'data/5_Clusters_1_SubCluster_PCA/Eval'

In [12]:
main(input_jsonl="output/weather_zeroshot_mistral_results.jsonl", output_csv="data/output_ekstrak_zeroshot_mistral.csv", model="mistral")

[OK] line 1: extracted 4 record(s)
[OK] line 2: extracted 4 record(s)
[OK] line 3: extracted 4 record(s)
[OK] line 4: extracted 3 record(s)
[OK] line 5: extracted 4 record(s)
[OK] line 6: extracted 2 record(s)
[OK] line 7: extracted 5 record(s)
[OK] line 8: extracted 1 record(s)
[OK] line 9: extracted 3 record(s)
[OK] line 10: extracted 5 record(s)
Saved 35 records to data/output_ekstrak_zeroshot_mistral.csv


In [20]:
main(input_jsonl="output/weather_zeroshot_qwen_results.jsonl", output_csv="data/output_ekstrak_zeroshot_qwen2.csv", model="mistral")

[OK] line 1: extracted 4 record(s)
[OK] line 2: extracted 4 record(s)
[OK] line 3: extracted 4 record(s)
[OK] line 4: extracted 4 record(s)
[OK] line 5: extracted 4 record(s)
[OK] line 6: extracted 2 record(s)
[OK] line 7: extracted 5 record(s)
[OK] line 8: extracted 3 record(s)
[OK] line 9: extracted 3 record(s)
[OK] line 10: extracted 5 record(s)
Saved 38 records to data/output_ekstrak_zeroshot_qwen2.csv


In [15]:
main(input_jsonl="output/weather_fewshot_mistral_results.jsonl", output_csv="data/output_ekstrak_fewshot_mistral.csv", model="mistral")

[OK] line 1: extracted 4 record(s)
[OK] line 2: extracted 4 record(s)
[OK] line 3: extracted 4 record(s)
[OK] line 4: extracted 4 record(s)
[OK] line 5: extracted 4 record(s)
[OK] line 6: extracted 2 record(s)
[OK] line 7: extracted 5 record(s)
[OK] line 8: extracted 3 record(s)
[OK] line 9: extracted 3 record(s)
[OK] line 10: extracted 5 record(s)
Saved 38 records to data/output_ekstrak_fewshot_mistral.csv


In [7]:
file_path = "data/weather/100_random_test_data.jsonl"
output_path = "data/5_Clusters_1_SubCluster_PCA/Eval/ground_truth_all_100_random_test_data.csv"

data = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))


all_records = []

for item in data:
    if "ground_truth" in item:
        all_records.extend(item["ground_truth"])

df = pd.DataFrame(all_records)
df.to_csv(f"{output_path}", index=False, float_format="%.2f")

print(f"CSV berhasil dibuat: {output_path}")


CSV berhasil dibuat: data/5_Clusters_1_SubCluster_PCA/Eval/ground_truth_all_100_random_test_data.csv


## CALCULATING THE L1LOSS FUNCTION



In [5]:
# From csv ground truth ke# L1 loss pipeline for multiple models vs. ground truth
# - Reads ground truth and LLM extraction CSVs from data/
# - Normalizes column names & units with robust heuristics
# - Joins on datetime (and city/province if available)
# - Computes per-variable MAE (L1) and overall average per model
# - Saves summary CSV and per-model joined comparisons for inspection

from pathlib import Path
import pandas as pd
import numpy as np
import json

DATA_DIR = Path("data/5_Clusters_1_SubCluster_PCA/Results")
OUTPUT_DIR = Path("data/5_Clusters_1_SubCluster_PCA/Eval")

CLUSTER_PART = Path("initial_prompts/5_Clusters_1_SubCluster_PCA")
GROUND_TRUTH_PATH = DATA_DIR / "ground_truth_all_test_data.csv"
MODEL_FILES = {
    "fewshot_llama": DATA_DIR / "output_ekstrak_fewshot_llama.csv",
    "fewshot_qwen": DATA_DIR / "output_ekstrak_fewshot_qwen.csv",
    "fewshot_mistral": DATA_DIR / "output_ekstrak_fewshot_mistral.csv",
    # "zeroshot_llama": DATA_DIR / "output_ekstrak_zeroshot_llama.csv",
    # "zeroshot_qwen": DATA_DIR / "output_ekstrak_zeroshot_qwen.csv",
    # "zeroshot_mistral": DATA_DIR / "output_ekstrak_zeroshot_mistral.csv",
}

# ---------- helpers ----------

def _first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def normalize_datetime(df):
    dt_col = _first_existing_col(df, [
        "datetime_iso", "datetime", "valid_time", "time", "timestamp", "date_time"
    ])
    if dt_col is None:
        for c in df.columns:
            if df[c].astype(str).str.contains(
                r"\d{4}-\d{2}-\d{2}.*\d{2}:\d{2}", regex=True
            ).any():
                dt_col = c
                break
    if dt_col is None:
        raise ValueError("No datetime-like column found")

    # --- minimal cleanup (2 baris) ---
    s = (df[dt_col].astype(str).str.strip()
         .str.replace(r'[\u200B\u200C\u200D\u2060\uFEFF\u00A0\u202F]', ' ', regex=True))

    # opsional: T -> spasi (boleh dihapus juga, pandas bisa parse 'T')
    s = s.str.replace("T", " ", regex=False)

    # --- parse bertahap: generic dulu, lalu fallback ke format eksplisit ---
    dt = pd.to_datetime(s, errors="coerce")  # TANPA utc=True

    if dt.isna().any():
        m = dt.isna()
        # paket format yang menutup kasus Qwen & GT kamu
        for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M"):
            dt2 = pd.to_datetime(s[m], format=fmt, errors="coerce")
            dt.loc[m] = dt2
            m = dt.isna()
            if not m.any():
                break

    # samakan presisi & bentuk dengan GT
    dt = dt.dt.floor("min")
    try:
        dt = dt.dt.tz_localize(None)  # buang tz bila ada
    except TypeError:
        pass

    out = df.copy()
    out["__dt__"] = dt
    return out

def normalize_location(df):
    # Attempt to standardize city/province if present
    city_col = _first_existing_col(df, ["city", "kota", "location_city"])
    prov_col = _first_existing_col(df, ["province", "provinsi", "state", "region", "location_province"])
    loc_col = _first_existing_col(df, ["location", "lokasi"])
    df = df.copy()
    if city_col is not None:
        df["__city__"] = df[city_col].astype(str).str.strip()
    elif loc_col is not None:
        # Try to split "City, Province"
        parts = df[loc_col].astype(str).str.split(",", n=1, expand=True)
        df["__city__"] = parts[0].str.strip()
        if prov_col is None and parts.shape[1] == 2:
            df["__province__"] = parts[1].str.strip()
    if prov_col is not None:
        df["__province__"] = df[prov_col].astype(str).str.strip()
    return df

def map_columns_to_canonical(df, is_ground_truth=False):
    """
    Try to map available columns to canonical names.
    Canonical: u10_ms, v10_ms, dewpoint_c, temperature_c, p_msl_hpa, p_surf_hpa, precip_m
    """
    df = df.copy()
    colmap = {}
    cols = {c.lower(): c for c in df.columns}

    def has(*names):
        for n in names:
            if n in cols:
                return cols[n]
        return None

    # u component
    colmap["u10_ms"] = has("u10_ms", "u10", "u_10m", "east_west_wind", "u_component", "u")
    # v component
    colmap["v10_ms"] = has("v10_ms", "v10", "v_10m", "north_south_wind", "v_component", "v")
    # temperature
    colmap["temperature_c"] = has("temperature_c", "temp_c", "temperature", "t2m_c", "t2m", "t")
    # dewpoint
    colmap["dewpoint_c"] = has("dewpoint_c", "dewpoint", "dewpoint_temperature", "td_c", "td2m", "d2m", "td")
    # mslp
    colmap["p_msl_hpa"] = has("p_msl_hpa", "msl", "mslp", "pressure_mean_sea_level", "pmsl", "p_msl")
    # surface pressure
    colmap["p_surf_hpa"] = has("p_surf_hpa", "sp", "surface_pressure", "ps", "p_surf", "psfc")
    # total precip (meters)
    colmap["precip_m"] = has("precip_m", "total_precipitation", "tp", "precipitation", "pr", "prcp")

    # Build a new dataframe with only present columns mapped
    mapped = {}
    for k, v in colmap.items():
        if v is not None:
            mapped[k] = df[v]

    out = df.copy()
    # Overwrite/insert canonical-named columns if present
    for k, series in mapped.items():
        out[k] = series

    return out, [k for k, v in colmap.items() if v is not None]

def detect_and_fix_units(df, source_label=""):
    """
    Heuristic unit normalization:
    - temperature_c: if mean > 200 => Kelvin -> convert to C
    - dewpoint_c: same heuristic
    - p_msl_hpa, p_surf_hpa: if mean > 2000 => Pa -> divide by 100
    - precip_m: if mean > 5 and max > 20 => assume mm -> /1000
    """
    df = df.copy()
    notes = []

    def _maybe_convert_temp(col):
        nonlocal df, notes
        if col in df.columns:
            s = pd.to_numeric(df[col], errors="coerce")
            if s.notna().any():
                m = s.mean(skipna=True)
                if m > 200:  # Kelvin?
                    df[col] = s - 273.15
                    notes.append(f"{source_label}:{col} assumed Kelvin -> converted to C")
                else:
                    df[col] = s

    def _maybe_convert_pressure(col):
        nonlocal df, notes
        if col in df.columns:
            s = pd.to_numeric(df[col], errors="coerce")
            if s.notna().any():
                m = s.mean(skipna=True)
                if m > 2000:  # Pa
                    df[col] = s / 100.0
                    notes.append(f"{source_label}:{col} assumed Pa -> converted to hPa")
                else:
                    df[col] = s

    def _maybe_convert_precip(col):
        nonlocal df, notes
        if col in df.columns:
            s = pd.to_numeric(df[col], errors="coerce")
            if s.notna().any():
                mx = s.max(skipna=True)
                mean = s.mean(skipna=True)
                # Heuristic: if typical values look like millimeters
                if (mean > 5 and mx > 20) or (mx > 100):  # likely mm
                    df[col] = s / 1000.0
                    notes.append(f"{source_label}:{col} assumed mm -> converted to meters")
                else:
                    df[col] = s

    # Wind components in m/s (assume already m/s, but cast to numeric)
    for col in ["u10_ms", "v10_ms"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in ["temperature_c", "dewpoint_c"]:
        _maybe_convert_temp(col)

    for col in ["p_msl_hpa", "p_surf_hpa"]:
        _maybe_convert_pressure(col)

    _maybe_convert_precip("precip_m")

    return df, notes

def reduce_duplicates(df):
    """
    If multiple rows per key (datetime, city, province), aggregate by mean.
    """
    key_cols = ["__dt__"]
    if "__city__" in df.columns:
        key_cols.append("__city__")
    if "__province__" in df.columns:
        key_cols.append("__province__")
    # Identify value columns (numeric canonical)
    val_cols = [c for c in ["u10_ms","v10_ms","dewpoint_c","temperature_c","p_msl_hpa","p_surf_hpa","precip_m"] if c in df.columns]
    if not val_cols:
        return df
    grp = df.groupby(key_cols, dropna=False)[val_cols].mean().reset_index()
    # Bring back string columns for city/province in a normalized display
    return grp

def prepare_dataframe(path, is_ground_truth=False, source_label=""):
    df = pd.read_csv(path)
    df = normalize_datetime(df)
    df = normalize_location(df)
    df, present = map_columns_to_canonical(df, is_ground_truth=is_ground_truth)
    df, notes = detect_and_fix_units(df, source_label=source_label)
    df = reduce_duplicates(df)
    return df, present, notes

def join_gt_pred(gt_df, pred_df):
    # Join strategy: prefer join on datetime + city + province if pred has them

    on_cols = ["__dt__"]
    for c in ["__city__", "__province__"]:
        if (c in pred_df.columns and c in gt_df.columns and pred_df[c].astype(str).str.strip().ne("").any() and gt_df[c].astype(str).str.strip().ne("").any()):
            on_cols.append(c)
    merged = pd.merge(gt_df, pred_df, on=on_cols, how="left", suffixes=("_gt", "_pred"),indicator=True)
    return merged, on_cols

def compute_l1_mae(joined_df):

    penalty_mode = "baseline_mean"
    baseline_const=0.0

    results = {}
    var_pairs = [
        ("u10_ms_gt","u10_ms_pred"),
        ("v10_ms_gt","v10_ms_pred"),
        ("dewpoint_c_gt","dewpoint_c_pred"),
        ("temperature_c_gt","temperature_c_pred"),
        ("p_msl_hpa_gt","p_msl_hpa_pred"),
        ("p_surf_hpa_gt","p_surf_hpa_pred"),
        ("precip_m_gt","precip_m_pred"),
    ]

    all_abs_errors = []

    for gcol, pcol in var_pairs:
        if gcol not in joined_df.columns:
            continue
        gt_vals = joined_df[gcol].astype(float)
        pred_vals = joined_df[pcol].astype(float) if pcol in joined_df.columns else pd.Series(np.nan, index=joined_df.index)

        if penalty_mode == "zero":
            penalty_val = 0.0
        elif penalty_mode == "baseline_const":
            penalty_val = float(baseline_const)
        elif penalty_mode == "baseline_mean":
            penalty_val = float(gt_vals.mean(skipna=True))
        else:
            raise ValueError("penalty_mode harus salah satu dari: zero, baseline_const, baseline_mean")

        pred_filled = pred_vals.where(~pred_vals.isna(), other=penalty_val)

        abs_err = (gt_vals - pred_filled).abs()

        results[gcol.replace("_gt","")] = float(abs_err.mean(skipna=True))

        all_abs_errors.extend(abs_err.dropna().tolist())

    if all_abs_errors:
        results["overall_mae"] = float(np.mean(all_abs_errors))

    return results

# ---------- run pipeline ----------

# Load and normalize ground truth
gt_df, gt_present, gt_notes = prepare_dataframe(GROUND_TRUTH_PATH, is_ground_truth=True, source_label="GT")
unit_notes = list(gt_notes)

summaries = []
per_model_joined_paths = {}

name_m = ""
for model_name, csv_path in MODEL_FILES.items():
    pred_df, pred_present, pred_notes = prepare_dataframe(csv_path, is_ground_truth=False, source_label=model_name)
    unit_notes.extend(pred_notes)

    joined, on_cols = join_gt_pred(gt_df, pred_df)

    name_m = model_name
    # Save joined for inspection
    out_joined_path = OUTPUT_DIR / f"joined_{model_name}.csv"
    joined.to_csv(out_joined_path, index=False)
    per_model_joined_paths[model_name] = str(out_joined_path)

    metrics = compute_l1_mae(joined)
    pred_cols = [c for c in joined.columns if c.endswith("_pred")]

    has_pred = joined[pred_cols].notna().any(axis=1)
    n_matched_rows = int((joined["_merge"] == "both").sum())

    coverage = (n_matched_rows / len(gt_df) * 100) if len(gt_df) > 0 else 0.0

    row = {"model": model_name, **metrics,
           "n_matched_rows": n_matched_rows,
           "total_rows": int(len(gt_df)),
           "coverage": coverage,
           "join_keys": json.dumps(on_cols)}
    summaries.append(row)

summary_df = pd.DataFrame(summaries).fillna("")
summary_path = OUTPUT_DIR / f"l1loss_summary_{name_m}.csv"
summary_df.to_csv(summary_path, index=False)

# Also show which canonical columns were present in GT and each model
present_table = {
    "source": ["ground_truth"] + list(MODEL_FILES.keys()),
    "present_vars": [", ".join(gt_present)]
}
for model_name, csv_path in MODEL_FILES.items():
    pred_df, pred_present, _ = prepare_dataframe(csv_path, is_ground_truth=False, source_label=model_name)
    present_table["present_vars"].append(", ".join(pred_present))

present_df = pd.DataFrame(present_table)


print("Saved summary to:", summary_path)
print("Joined files:", json.dumps(per_model_joined_paths, indent=2))
print("\nUnit conversion notes:\n- " + "\n- ".join(unit_notes) if unit_notes else "No unit conversions were applied.")


Saved summary to: logs/L1_LOSS/l1loss_summary_zeroshot_mistral.csv
Joined files: {
  "zeroshot_llama": "logs/L1_LOSS/joined_zeroshot_llama.csv",
  "zeroshot_qwen": "logs/L1_LOSS/joined_zeroshot_qwen.csv",
  "zeroshot_mistral": "logs/L1_LOSS/joined_zeroshot_mistral.csv"
}
No unit conversions were applied.


In [30]:
import json

with open("data/weather/10_test_data.jsonl", "r") as f:
    test_data = [json.loads(i) for i in f]

print(test_data[0]["ground_truth"])


{'2012-12-22 13:00:00': {'east_west_wind_speed_10m': 0.37, 'north_south_wind_speed_10m': 0.23, 'dewpoint_temperature_2m': 15.72, 'air_temperature_2m': 16.59, 'mean_sea_level_pressure': 1012.15, 'surface_pressure': 820.79, 'total_precipitation': 0.005, 'latitude': -4.5, 'longitude': 140.5, 'area': 'Pegunungan Bintang, Highland Papua'}, '2012-12-22 14:00:00': {'east_west_wind_speed_10m': 0.39, 'north_south_wind_speed_10m': 0.24, 'dewpoint_temperature_2m': 15.96, 'air_temperature_2m': 16.65, 'mean_sea_level_pressure': 1012.03, 'surface_pressure': 820.64, 'total_precipitation': 0.007, 'latitude': -4.5, 'longitude': 140.5, 'area': 'Pegunungan Bintang, Highland Papua'}, '2012-12-22 15:00:00': {'east_west_wind_speed_10m': 0.17, 'north_south_wind_speed_10m': 0.7, 'dewpoint_temperature_2m': 15.64, 'air_temperature_2m': 16.35, 'mean_sea_level_pressure': 1011.68, 'surface_pressure': 820.38, 'total_precipitation': 0.0101, 'latitude': -4.5, 'longitude': 140.5, 'area': 'Pegunungan Bintang, Highland 

In [10]:
import os
import glob
import pandas as pd

csv_files = os.path.join("logs/", "weather_mistral:7b-instruct-v0.2-q2_K_experiment_log_20251015_204223.csv")
# Gabungkan menjadi path lengkap
# Baca file CSV
df = pd.read_csv(csv_files)

# --- 1. Hitung jumlah baris ---
row_count = len(df)
print(f"Jumlah baris pada file: {row_count}")



Jumlah baris pada file: 101


In [17]:
# --- 2. Konversi kolom 'LLM Result in sec' ke float ---
df['LLM Result in sec'] = df['LLM Result in sec'].astype(float)

# --- 3. Hitung jumlah (SUM) dari kolom 'LLM Result in sec' mulai baris ke-2 hingga akhir ---
# (baris pertama = indeks 0, jadi mulai dari indeks 1)
sum_llm = df.loc[1:, 'LLM Result in sec'].sum()
avg = df.loc[1:, 'LLM Result in sec'].mean()
print(f"Total SUM dari baris ke-2 sampai akhir kolom 'LLM Result in sec': {sum_llm}")
print(f"Total avg dari baris ke-2 sampai akhir kolom 'LLM Result in sec': {avg}")



Total SUM dari baris ke-2 sampai akhir kolom 'LLM Result in sec': 22485.469804329005
Total avg dari baris ke-2 sampai akhir kolom 'LLM Result in sec': 227.1259576194849


In [16]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
final_avg_rows = df[df.apply(lambda row: row.astype(str).str.contains('Final Avg.', case=False).any(), axis=1)]
print(final_avg_rows)

      Challenge Prompt Ground Truth LLM Result LLM Result (Clean)  \
100  Final Avg.    NaN          NaN        NaN                NaN   

     Levenshtein Distance  Normalized Levenshtein Distance    LCS      NLCS  \
100                190.48                         0.165418  32.99  0.032648   

       MAE(L1)   COVERAGE  BertScore  CosineSimilarity      BLEU   ROUGE-L  \
100  52.731765  82.154762   0.718536          0.584526  0.095955  0.701991   

     STRICT   LENIENT                          Model  Operation  Mode  \
100     1.0  0.292562  mistral:7b-instruct-v0.2-q2_K        NaN   NaN   

    Prompting  LLM Result in sec  
100   fewshot                NaN  


In [18]:
import pandas as pd
import os, glob
from sklearn.feature_extraction.text import CountVectorizer


#Extraction of total inference time and token counts from multiple CSV files
path_folder = "/Users/pangaribuan.b/WeatherLLM/tamp"
csv_paths = glob.glob(os.path.join(path_folder, "*.csv"))

#counting tokens total per column
def count_tokens(text_series):
    return sum(len(str(t).split()) for t in text_series.fillna(""))

results = []

# looping through each csv file
for path in csv_paths:
    df = pd.read_csv(path)
    print("cek")
    df["LLM Result in sec"] = pd.to_numeric(df["LLM Result in sec"], errors="coerce")
    print(df["LLM Result in sec"])
    # Hitung total waktu mulai baris ke-2
    total_time = df["LLM Result in sec"].iloc[1:].sum()

    # Hitung jumlah token Prompt dan LLM Result
    total_prompt_tokens = count_tokens(df["Prompt"])
    total_llm_tokens = count_tokens(df["LLM Result"])

    # Ambil info model dan prompting
    model = df["Model"].iloc[0] if "Model" in df.columns else os.path.basename(path)
    prompting = df["Prompting"].iloc[0] if "Prompting" in df.columns else "unknown"

    results.append({
        "File": os.path.basename(path),
        "Model": model,
        "Prompting": prompting,
        "Total Inference Time (s, rows 2-end)": total_time,
        "Total Prompt Tokens": total_prompt_tokens,
        "Total LLM Result Tokens": total_llm_tokens
    })

summary_df = pd.DataFrame(results)


['/Users/pangaribuan.b/WeatherLLM/tamp/weather_qwen2.5:7b-instruct-q2_K_experiment_log_20251011_103039.csv', '/Users/pangaribuan.b/WeatherLLM/tamp/weather_llama3:8b-instruct-q2_K_experiment_log_20251011_101641.csv', '/Users/pangaribuan.b/WeatherLLM/tamp/weather_mistral:7b-instruct-v0.2-q2_K_experiment_log_20251009_145324.csv', '/Users/pangaribuan.b/WeatherLLM/tamp/weather_qwen2.5:7b-instruct-q2_K_experiment_log_20251009_140644.csv', '/Users/pangaribuan.b/WeatherLLM/tamp/weather_mistral:7b-instruct-v0.2-q2_K_experiment_log_20251011_094120.csv', '/Users/pangaribuan.b/WeatherLLM/tamp/weather_mistral:7b-instruct-v0.2-q2_K_experiment_log_20251009_121531.csv', '/Users/pangaribuan.b/WeatherLLM/tamp/weather_mistral:7b-instruct-v0.2-q2_K_experiment_log_20251011_113739.csv', '/Users/pangaribuan.b/WeatherLLM/tamp/weather_qwen2.5:7b-instruct-q2_K_experiment_log_20251009_175806.csv', '/Users/pangaribuan.b/WeatherLLM/tamp/weather_llama3:8b-instruct-q2_K_experiment_log_20251009_161820.csv', '/Users/p

In [19]:
summary_df

,File,Model,Prompting,"Total Inference Time (s, rows 2-end)",Total Prompt Tokens,Total LLM Result Tokens
0,weather_qwen2.5:7b-instruct-q2_K_experiment_lo...,qwen2.5:7b-instruct-q2_K,zeroshot,1804.103023,23567,10537
1,weather_llama3:8b-instruct-q2_K_experiment_log...,llama3:8b-instruct-q2_K,zeroshot,1040.484373,23567,5460
2,weather_mistral:7b-instruct-v0.2-q2_K_experime...,mistral:7b-instruct-v0.2-q2_K,fewshot,8376.461863,23567,9024
3,weather_qwen2.5:7b-instruct-q2_K_experiment_lo...,qwen2.5:7b-instruct-q2_K,fewshot,8164.023046,13624,4911
4,weather_mistral:7b-instruct-v0.2-q2_K_experime...,mistral:7b-instruct-v0.2-q2_K,zeroshot,2236.936833,23567,9638
5,weather_mistral:7b-instruct-v0.2-q2_K_experime...,mistral:7b-instruct-v0.2-q2_K,fewshot,8076.460126,13624,7108
6,weather_mistral:7b-instruct-v0.2-q2_K_experime...,mistral:7b-instruct-v0.2-q2_K,zeroshot,6043.216760,13624,5627
7,weather_qwen2.5:7b-instruct-q2_K_experiment_lo...,qwen2.5:7b-instruct-q2_K,fewshot,8327.564862,22727,9245
8,weather_llama3:8b-instruct-q2_K_experiment_log...,llama3:8b-instruct-q2_K,fewshot,5876.949242,23567,10057
9,weather_qwen2.5:7b-instruct-q2_K_experiment_lo...,qwen2.5:7b-instruct-q2_K,zeroshot,2129.149423,13624,4443
